# Solar Active-Region Detection — Kaggle ROLLING run, 13 CHANNELS (max memory)

**One cell to run** (safe to re-run any time).

Right-hand panel (**Session options**): **Internet ON**, **GPU T4 x2** (both used
automatically), **Persistence: Files only** (keeps data + model across sessions).

## What this run does

Full **13-channel** input (8 AIA EUV + 5 HMI field components) on a **rolling
window of the newest 200 frames** (13-channel tiles are ~46 MB, so 200 frames
is what fits Kaggle's 20 GiB wall): each batch retires the oldest frames and
reuses their disk for new ones. Guarantees:

- **each frame trains ~2-4 passes before it may be retired** (3 h minimum
  lifetime) — it is *used*, not just streamed past
- **retired frame names are recorded forever** — no frame is ever
  re-downloaded; over days the model sees thousands of unique frames
- **no errors from rotation** — a tile freed mid-epoch becomes a neutral
  background sample; rotation failures are logged and ignored (data stays safe)
- **validation frames are never retired** (they are the scoreboard)
- **when the 70,823-frame index is finally exhausted (~2 weeks of downloading),
  rotation stops automatically** and the model keeps re-practicing what's on disk

## The memory stack (anti-forgetting for rotating data)

- **BASE_CHANNELS=64** — 33M params (2x the 48-base): more capacity to store
  what it has seen (batch auto-shrinks to fit each T4)
- **LR=1e-4** — a third of the default learning rate: new updates overwrite
  less of the old knowledge
- **EMA (built in)** — best.pt is a long-term average of the weights

Newest model is mirrored to /kaggle/output (downloadable, Output tab) every
5 min. When a session ends (12 h): run the same cell again — it resumes.

In [ ]:
%%bashset -xexport HOME=/kaggle/workingcd /kaggle/working# 1) Stop any previous run (lock PID + trainer + downloader), clean orphaned temp files:P=$(cat /kaggle/working/solar_results/arpil/run_forever.lock 2>/dev/null)[ -n "$P" ] && kill -TERM "$P" 2>/dev/nullpkill -f "SOALR/scripts/train_streaming.py" 2>/dev/nullpkill -f "SOALR/scripts/build_arpil_resumable.py" 2>/dev/nullsleep 5pkill -9 -f "SOALR/scripts/train_streaming.py" 2>/dev/nullpkill -9 -f "SOALR/scripts/build_arpil_resumable.py" 2>/dev/nullrm -rf /tmp/arpil_resume_* 2>/dev/null# 2) One-time channel-set check: this preset needs 13-channel data. A previous#    run with a different channel set (e.g. the 3-channel attempt) is#    incompatible (its frames would count as "done" and never re-tile for 13#    channels), so reset the data dir. Same-set 13-channel data is kept.NCH=$(ls /kaggle/working/solar_data/arpil/images 2>/dev/null | wc -l)if [ "$NCH" != "13" ]; then    echo "resetting data dir for the 13-channel preset (found ${NCH:-0} channel dirs)"    rm -rf /kaggle/working/solar_datafi# 3) Fresh code repo (tiny, ~15 s):rm -rf /kaggle/work SOALRgit clone -q -b arena/01a04247-soalr-active-region-detection \    https://github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-.git SOALR \|| { mkdir -p SOALR && wget -qO /tmp/repo.tgz \    https://codeload.github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-/tar.gz/refs/heads/arena/01a04247-soalr-active-region-detection \    && tar xzf /tmp/repo.tgz -C SOALR --strip-components=1; }if [ ! -x SOALR/scripts/run_forever.sh ]; then    echo "STOP: repository download failed. In the notebook Settings, make sure Internet is ON, then re-run this cell."    exit 1ficd SOALR# 4) NO venv on Kaggle (ensurepip is broken there); torch is preinstalled:python3 -m pip install -q -r requirements.txtpython3 -m pip cache purge 2>/dev/nullpython3 -c "import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())"# 5) Mirror the newest model + log to /kaggle/output every 5 min (downloadable#    from the Output tab after a session ends):mkdir -p /kaggle/output(    while :; do        sleep 300        cp -f /kaggle/working/solar_results/arpil/continuous/best.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/last.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/metrics.jsonl /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/run_forever.log      /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/STATUS.md            /kaggle/output/ 2>/dev/null    done) &SAVE_WATCHER=$!# 6) 13-CHANNEL ROLLING MAX-MEMORY preset:#    ROLLING=1 ROLLING_WINDOW=200          newest-200-frames window (13ch tiles#                                          are ~46 MB: 200 x 46 MB + in-flight#                                          downloads + overhead peak ~17.5 GiB,#                                          under the 20 GiB wall)#    ROLLING_MIN_LIFETIME_HOURS=3          each frame trains ~2-4 passes before#                                          it may be retired ("use it 2x first")#    TILE_GRACE_HOURS=0                    free retired tiles immediately; a#                                          mid-epoch reference to a freed tile#                                          becomes a background sample (no error)#    MAX_TOTAL_FRAMES=0                    download NEW frames forever (rotation,#                                          not a cap, sizes the disk); when the#                                          70,823-frame index is exhausted, rotation#                                          stops and it re-practices what's on disk#    BASE_CHANNELS=64 (33M params) + LR=1e-4 + EMA = the anti-forgetting stackCHANNELS="aia94 aia131 aia1600 aia171 aia193 aia211 aia304 aia335 hmi_m hmi_bx hmi_by hmi_bz hmi_v" \SOLAR_PYTHON="$(command -v python3)" BASE_CHANNELS=64 DEEP_SUPERVISION=1 \ROLLING=1 ROLLING_WINDOW=200 ROLLING_MIN_LIFETIME_HOURS=3 TILE_GRACE_HOURS=0 LR=1e-4 \MIN_FREE_GB=4 MAX_TOTAL_FRAMES=0 FRAMES_PER_CYCLE=200 DOWNLOAD_WORKERS=4 \CPU_HEADROOM=0 TILES_PER_EPOCH=200 VAL_EPOCH=10 VAL_SUBSET=300 \bash scripts/run_forever.shkill "$SAVE_WATCHER" 2>/dev/null

## What you will see

- `torch 2.x.x | CUDA available: True | GPUs: 2` — environment healthy
- Preflight `3 ok`, then `Downloading 200 frames with 4 parallel workers ...`
- `[stream] using 2 GPUs with DataParallel (each batch splits across both)`
- `[stream] epoch=0001 loss=0.7x ...` — finite loss; first `val_dice` at epoch 10
- From ~3 h in, a line each batch like
  `[rolling] 300 -> 200 frames on disk (retired 100 this batch; 100 retired total; 100 old tiles freed)`

## Re-running

Idempotent: stops any running instance, re-clones the code (~15 s), resumes.
With **Persistence: Files only**, data + the retired name record + checkpoints
survive — no frame is ever downloaded twice.